In [ ]:
FPS = 30
FFT_WINDOW_SECONDS = 0.25 # how many seconds of audio make up an FFT window

# Note range to display
FREQ_MIN = 10
FREQ_MAX = 1000

# Notes to display
TOP_NOTES = 3

# Names of the notes
NOTE_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

# Output size. Generally use SCALE for higher res, unless you need a non-standard aspect ratio.
RESOLUTION = (1920, 1080)
SCALE = 2 # 0.5=QHD(960x540), 1=HD(1920x1080), 2=4K(3840x2160)


[notice] A new release of pip is available: 24.1.2 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# !ffmpeg -ss 50 -i muthai_tharu.mp3 -c copy muthai_tharu1.mp3
# !ffmpeg -i happy_birthday.mp3 happy_birthday.wav

In [5]:
import matplotlib.pyplot as plt
from scipy.fftpack import fft
from scipy.io import wavfile # get the api
import os


AUDIO_FILE = "muthai_tharu.wav"

fs, data = wavfile.read(AUDIO_FILE) # load the data
audio = data.T[0] # this is a two channel soundtrack, get the first track
FRAME_STEP = (fs / FPS) # audio samples per video frame
FFT_WINDOW_SIZE = int(fs * FFT_WINDOW_SECONDS)
AUDIO_LENGTH = len(audio)/fs

In [7]:
import numpy as np


# See https://newt.phys.unsw.edu.au/jw/notes.html
def freq_to_number(f): return 69 + 12*np.log2(f/440.0)
def number_to_freq(n): return 440 * 2.0**((n-69)/12.0)
def note_name(n): return NOTE_NAMES[n % 12] + str(int(n/12 - 1))

# Hanning window function
window = 0.5 * (1 - np.cos(np.linspace(0, 2*np.pi, FFT_WINDOW_SIZE, False)))

xf = np.fft.rfftfreq(FFT_WINDOW_SIZE, 1/fs)
FRAME_COUNT = int(AUDIO_LENGTH*FPS)
FRAME_OFFSET = int(len(audio)/FRAME_COUNT)




In [8]:
import plotly.graph_objects as go

def plot_fft(p, xf, fs, notes, dimensions=(960,540)):
  layout = go.Layout(
      title="frequency spectrum",
      autosize=False,
      width=dimensions[0],
      height=dimensions[1],
      xaxis_title="Frequency (note)",
      yaxis_title="Magnitude",
      font={'size' : 24}
  )

  fig = go.Figure(layout=layout,
                  layout_xaxis_range=[FREQ_MIN,FREQ_MAX],
                  layout_yaxis_range=[0,1]
                  )
  
  fig.add_trace(go.Scatter(
      x = xf,
      y = p))
  
  for note in notes:
    fig.add_annotation(x=note[0]+10, y=note[2],
            text=note[1],
            font = {'size' : 48},
            showarrow=False)
  return fig

def extract_sample(audio, frame_number):
  end = frame_number * FRAME_OFFSET
  begin = int(end - FFT_WINDOW_SIZE)

  if end == 0:
    # We have no audio yet, return all zeros (very beginning)
    return np.zeros((np.abs(begin)),dtype=float)
  elif begin<0:
    # We have some audio, padd with zeros
    return np.concatenate([np.zeros((np.abs(begin)),dtype=float),audio[0:end]])
  else:
    # Usually this happens, return the next sample
    return audio[begin:end]

def find_top_notes(fft,num):
  if np.max(fft.real)<0.001:
    return []

  lst = [x for x in enumerate(fft.real)]
  lst = sorted(lst, key=lambda x: x[1],reverse=True)

  idx = 0
  found = []
  found_note = set()
  while( (idx<len(lst)) and (len(found)<num) ):
    f = xf[lst[idx][0]]
    y = lst[idx][1]
    n = freq_to_number(f)
    n0 = int(round(n))
    name = note_name(n0)

    if name not in found_note:
      found_note.add(name)
      s = [f,note_name(n0),y]
      found.append(s)
    idx += 1
    
  return found

In [9]:
# Pass 1, find out the maximum amplitude so we can scale.
mx = 0
for frame_number in range(FRAME_COUNT):
  sample = extract_sample(audio, frame_number)

  fft = np.fft.rfft(sample * window)
  fft = np.abs(fft).real 
  mx = max(np.max(fft),mx)

print(f"Max amplitude: {mx}")



Max amplitude: 42619422.55374495


In [10]:
import tqdm
import numpy as np
import matplotlib.pyplot as plt  # Using matplotlib for efficiency
import os

# Ensure the output directory exists
output_dir = "C:\\DEV\\SangeethamAI\\content"
os.makedirs(output_dir, exist_ok=True)
with open('music_notes.txt', 'w+') as musicfile:

    # Pass 2, produce the animation
    for frame_number in tqdm.tqdm(range(FRAME_COUNT)):
        #print(f"Processing frame {frame_number + 1}/{FRAME_COUNT}")  # Debug print

        # Extract the sample
        sample = extract_sample(audio, frame_number)

        # Compute FFT
        fft = np.fft.rfft(sample * window)
        fft = np.abs(fft) / mx 

        # Find top notes
        s = find_top_notes(fft, 4)

        # Plot FFT using matplotlib for faster rendering
        plt.figure()
        plt.plot(xf, fft.real)  # Assuming `xf` and `fft` dimensions match
        plt.title(f"FFT Frame {frame_number}")
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('Amplitude')
        plt.ylim(0,1)
        plt.xlim(0,1000)

        # Annotate the top notes on the FFT plot
        for note in s:
            frequency, note_label, magnitude = note
            #note_label = western_to_carnatic[note_label[:-1]]
            plt.annotate(note_label, 
                        xy=(frequency, magnitude), 
                        xytext=(frequency + 10, magnitude + 0.05), 
                        fontsize=9, color='red', 
                        arrowprops=dict(arrowstyle="->", color='red'))
            musicfile.write(note_label + " ")
        # # Save the frame
        frame_path = os.path.join(output_dir, f"frame{frame_number}.png")
        plt.savefig(frame_path, dpi=100)  # Lower DPI for faster saving
        plt.close()  # Close the figure to release memory

100%|██████████| 3932/3932 [04:49<00:00, 13.60it/s]


In [11]:
!ffmpeg -y -r {FPS} -f image2 -s 1920x1080 -i content/frame%d.png -i {AUDIO_FILE} -c:v libx264 -c:a aac -pix_fmt yuv420p -shortest movie.mp4


ffmpeg version 2024-07-18-git-fa5a605542-full_build-www.gyan.dev Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 13.2.0 (Rev5, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-libsnappy --enable-zlib --enable-librist --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-libbluray --enable-libcaca --enable-sdl2 --enable-libaribb24 --enable-libaribcaption --enable-libdav1d --enable-libdavs2 --enable-libopenjpeg --enable-libquirc --enable-libuavs3d --enable-libxevd --enable-libzvbi --enable-libqrencode --enable-librav1e --enable-libsvtav1 --enable-libvvenc --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxavs2 --enable-libxeve --enable-libxvid --enable-libaom --enable-libjxl --enable-libvpx --enable-mediafoundation --enable-libass --enable-fre